# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Lecture 8: Random Generation and Markov-Chain Simulation

This notebook accompanies Chapter 6, Section 6.1. We study finite-state
deterministic generators, distinguish a transient from a period, check a
full-period example, and see why uniform one-point frequencies do not imply
independence.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt


## A simple congruential generator

For $M\geq2$, a congruential generator maps
$S_M=\{0,\ldots,M-1\}$ to itself by

$$
u_{i+1}=(a u_i+b)\bmod M.
$$

Once $(a,b,M)$ and the seed $u_0$ are fixed, the sequence is completely
deterministic. The function below returns $u_0,\ldots,u_{n-1}$, so the
indexing convention is explicit.


In [ ]:
def lcg_states(a, b, modulus, seed, n):
    if modulus < 2 or n < 0:
        raise ValueError("modulus must be at least 2 and n must be nonnegative")
    if not 0 <= seed < modulus:
        raise ValueError("seed must belong to {0, ..., modulus-1}")
    states = np.empty(n, dtype=np.int64)
    state = int(seed)
    for i in range(n):
        states[i] = state
        state = (a * state + b) % modulus
    return states


print(lcg_states(a=3, b=0, modulus=16, seed=1, n=9))


## The start-up phase and the eventual cycle

Every deterministic map on a finite state space eventually revisits a state.
From then on it repeats. If the first repeated cycle begins at index $\mu$
and has length $T$, then the values before $\mu$ form the transient and
$T$ is the eventual period. A sequence need not be periodic from its
initial seed.


In [ ]:
def orbit_structure(step, seed):
    seen_at = {}
    path = []
    state = seed
    while state not in seen_at:
        seen_at[state] = len(path)
        path.append(state)
        state = step(state)
    transient_length = seen_at[state]
    period = len(path) - transient_length
    return path, transient_length, period, state


path, transient, period, repeated_state = orbit_structure(lambda x: x // 2, 15)
print("path before first repeat:", path)
print("transient length:", transient, "period:", period, "repeated state:", repeated_state)

step_lcg = lambda x: (3 * x) % 16
path, transient, period, repeated_state = orbit_structure(step_lcg, 1)
print("(3,0,16) from seed 1:", path, "transient:", transient, "period:", period)


## Choosing parameters for a full cycle

The Hull--Dobell theorem says that $(a,b,M)$ has period $M$ from every
seed exactly when:

1. $\gcd(b,M)=1$;
2. every prime divisor of $M$ divides $a-1$; and
3. if $4\mid M$, then $4\mid(a-1)$.

For $(a,b,M)=(5,1,16)$, these conditions hold. One complete cycle visits
all 16 states exactly once. This gives uniform one-point frequencies over a
cycle, but it does not give independent outputs.


In [ ]:
a, b, modulus = 5, 1, 16
periods = []
for seed in range(modulus):
    _, transient, period, _ = orbit_structure(
        lambda x, a=a, b=b, modulus=modulus: (a * x + b) % modulus,
        seed,
    )
    periods.append((transient, period))

cycle = lcg_states(a, b, modulus, seed=0, n=modulus)
print("Hull-Dobell gcd condition:", math.gcd(b, modulus) == 1)
print("all seeds have transient 0 and period 16:", all(item == (0, 16) for item in periods))
print("one cycle:", cycle)
print("each state appears once:", np.array_equal(np.sort(cycle), np.arange(modulus)))


Normalized values $w_i=u_i/M$ lie on the grid
$\{0,1/M,\ldots,(M-1)/M\}$, so they are always strictly less than 1.
Even for a full-period generator, every current state has exactly one
possible successor. Among the $M^2$ possible ordered pairs, only $M$
can occur in the cycle. Independent discrete uniform variables would assign
positive probability to all $M^2$ pairs.


In [ ]:
normalized = cycle / modulus
successors = np.roll(cycle, -1) / modulus

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(normalized, bins=np.arange(modulus + 1) / modulus,
             edgecolor="black")
axes[0].set(xlabel="$u_i/M$", ylabel="count", title="one-point frequencies")
axes[1].scatter(normalized, successors)
axes[1].set(xlabel="$u_i/M$", ylabel="$u_{i+1}/M$", title="successive pairs")
plt.tight_layout()
plt.show()


## A quick look at what went wrong with RANDU

RANDU used the multiplicative recurrence
$u_{i+1}=65539u_i\bmod 2^{31}$. Its states satisfy

$$
u_{i+2}-6u_{i+1}+9u_i=0\pmod{2^{31}},
$$

a rigid relation among successive triples. Matching marginal histograms
cannot detect all such dependence.


In [ ]:
randu_modulus = 2**31
randu = lcg_states(65539, 0, randu_modulus, seed=1, n=5000)
triple_relation = (randu[2:] - 6 * randu[1:-1] + 9 * randu[:-2]) % randu_modulus
print("all RANDU triples satisfy the modular relation:", np.all(triple_relation == 0))


## Using modern generators in practice

NumPy's `default_rng` provides a modern pseudorandom generator. A fixed seed
makes an experiment reproducible; changing the seed changes the deterministic
output stream. In mathematical analyses we often model high-quality outputs
as independent uniforms, but the software itself remains deterministic once
seeded.


In [ ]:
rng_a = np.random.default_rng(2026)
rng_b = np.random.default_rng(2026)
sample_a = rng_a.random(6)
sample_b = rng_b.random(6)
print(sample_a)
print("same seed reproduces the stream:", np.array_equal(sample_a, sample_b))


## Checkpoint: pseudorandom generators

1. For $(a,b,M)=(3,0,16)$, find the periods from seeds 0, 1, and 2.
2. Choose another mixed congruential generator with $M=32$ satisfying the
   Hull--Dobell conditions. Verify its period from every seed.
3. Count the distinct consecutive pairs in one full cycle and compare with
   the $M^2$ pairs possible under two independent uniforms on $S_M$.


## Turning pseudorandom numbers into samples

This notebook accompanies Chapter 6, Sections 6.2--6.3, and Chapter 7,
Sections 7.1--7.2. It implements inversion, accept--reject, and Box--Muller
sampling in Python, then simulates a finite Markov chain using fresh random
inputs at every step.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(808)


## Sampling with the inverse CDF

For a CDF $F$, the generalized inverse is

$$
F^{-1}(u)=\inf\{x\in\mathbb R:F(x)\geq u\},\qquad0<u<1.
$$

It remains valid when $F$ has jumps or flat parts. If
$U\sim\mathrm{Uniform}([0,1])$, then $F^{-1}(U)$ has CDF $F$.

Examples:

- Uniform on $[a,b]$: $F^{-1}(u)=a+(b-a)u$.
- Exponential with rate $\lambda>0$:
  $F^{-1}(u)=-\log(1-u)/\lambda$.
- Bernoulli with success probability $p$: return 0 for
  $0<u\leq1-p$ and 1 for $1-p<u<1$.


In [ ]:
n = 30_000
uniform_inputs = rng.random(n)

a, b = -2.0, 3.0
uniform_sample = a + (b - a) * uniform_inputs

rate = 2.0
exponential_sample = -np.log1p(-uniform_inputs) / rate

p = 0.30
bernoulli_sample = (uniform_inputs > 1 - p).astype(int)

print(f"Uniform sample mean {uniform_sample.mean():.3f}; theory {(a+b)/2:.3f}")
print(f"Exponential sample mean {exponential_sample.mean():.3f}; theory {1/rate:.3f}")
print(f"Bernoulli sample mean {bernoulli_sample.mean():.3f}; theory {p:.3f}")


In [ ]:
x_grid = np.linspace(0, 3.5, 500)
exponential_density = rate * np.exp(-rate * x_grid)

fig, ax = plt.subplots(figsize=(7, 3.7))
ax.hist(exponential_sample, bins=70, range=(0, 3.5), density=True,
        alpha=0.55, label="inversion sample")
ax.plot(x_grid, exponential_density, color="black", label="target density")
ax.set(xlabel="x", ylabel="density")
ax.legend()
plt.show()


## Sampling by accept--reject

Let $f$ be a target density, $g$ a proposal density, and suppose
$f(x)\leq M g(x)$ for every $x$, with $M\geq1$. Independently draw
$Y\sim g$ and $U\sim\mathrm{Uniform}([0,1])$, and accept $Y$ when

$$
U\leq\frac{f(Y)}{M g(Y)}.
$$

The accepted values have density $f$, and the acceptance probability is
$1/M$.

Here the target is $f(x)=\tfrac12\cos x$ on
$(-\pi/2,\pi/2)$, and the proposal is uniform on the same interval,
so $g(x)=1/\pi$, the smallest valid constant is $M=\pi/2$, and the
acceptance condition simplifies to $U\leq\cos Y$.


In [ ]:
def sample_cosine_target(size, generator):
    accepted = []
    proposals = 0
    while len(accepted) < size:
        proposal = generator.uniform(-np.pi / 2, np.pi / 2)
        uniform = generator.random()
        proposals += 1
        if uniform <= np.cos(proposal):
            accepted.append(proposal)
    return np.asarray(accepted), proposals


cosine_sample, proposal_count = sample_cosine_target(5000, rng)
acceptance_rate = cosine_sample.size / proposal_count
print(f"observed acceptance rate = {acceptance_rate:.4f}")
print(f"theoretical acceptance rate = {2/np.pi:.4f}")

cosine_grid = np.linspace(-np.pi / 2, np.pi / 2, 500)
fig, ax = plt.subplots(figsize=(7, 3.7))
ax.hist(cosine_sample, bins=60, density=True, alpha=0.55, label="accepted proposals")
ax.plot(cosine_grid, 0.5 * np.cos(cosine_grid), color="black", label="target density")
ax.set(xlabel="x", ylabel="density")
ax.legend()
plt.show()


## Sampling normal variables with Box--Muller

If $U_1,U_2$ are independent uniforms, then, except on the probability-zero
event $U_1=0$,

$$
Z_0=\sqrt{-2\log U_1}\cos(2\pi U_2),\qquad
Z_1=\sqrt{-2\log U_1}\sin(2\pi U_2)
$$

are independent standard normal random variables. We guard the logarithm in
code even though an exact zero from the generator is extraordinarily rare.


In [ ]:
n_pairs = 10_000
u1 = np.maximum(rng.random(n_pairs), np.finfo(float).tiny)
u2 = rng.random(n_pairs)
radius = np.sqrt(-2 * np.log(u1))
z0 = radius * np.cos(2 * np.pi * u2)
z1 = radius * np.sin(2 * np.pi * u2)
normal_sample = np.concatenate([z0, z1])

normal_grid = np.linspace(-4, 4, 500)
normal_density = np.exp(-normal_grid**2 / 2) / np.sqrt(2 * np.pi)
fig, ax = plt.subplots(figsize=(7, 3.7))
ax.hist(normal_sample, bins=70, range=(-4, 4), density=True,
        alpha=0.55, label="Box--Muller sample")
ax.plot(normal_grid, normal_density, color="black", label="standard normal density")
ax.set(xlabel="z", ylabel="density")
ax.legend()
plt.show()

print("sample correlation of paired outputs =", np.corrcoef(z0, z1)[0, 1])


## Simulating a small Markov chain

For the weather states $(D,W)$, use the row-stochastic transition matrix

$$
P=\begin{pmatrix}3/4&1/4\\1/2&1/2\end{pmatrix}.
$$

Given the current state $x$, inversion sampling applied to row $P(x,\cdot)$
chooses the next state. Under the mathematical model, a fresh independent
uniform is used at every step. Reusing one uniform value generally gives the
wrong multi-step law.


In [ ]:
states = np.array(["D", "W"])
transition = np.array([[0.75, 0.25],
                       [0.50, 0.50]])


def simulate_markov_chain(transition_matrix, initial_state, steps, generator):
    transition_matrix = np.asarray(transition_matrix, dtype=float)
    if (transition_matrix.ndim != 2
            or transition_matrix.shape[0] != transition_matrix.shape[1]
            or np.any(transition_matrix < 0)
            or not np.allclose(transition_matrix.sum(axis=1), 1)):
        raise ValueError("transition_matrix must be square and row-stochastic")
    if not 0 <= initial_state < transition_matrix.shape[0]:
        raise ValueError("initial_state is outside the state space")

    cumulative_rows = np.cumsum(transition_matrix, axis=1)
    path = np.empty(steps + 1, dtype=int)
    path[0] = initial_state
    for t in range(1, steps + 1):
        uniform = generator.random()  # a fresh input at this time step
        path[t] = np.searchsorted(cumulative_rows[path[t - 1]], uniform, side="right")
    return path


path = simulate_markov_chain(transition, initial_state=0, steps=20, generator=rng)
print("one path:", states[path])


In [ ]:
# Check the two-step distribution with many independent simulated paths.
repetitions = 20_000
end_states = np.empty(repetitions, dtype=int)
for repetition in range(repetitions):
    end_states[repetition] = simulate_markov_chain(
        transition, initial_state=0, steps=2, generator=rng
    )[-1]

empirical_distribution = np.bincount(end_states, minlength=2) / repetitions
theoretical_distribution = np.array([1.0, 0.0]) @ np.linalg.matrix_power(transition, 2)
print("simulated distribution at time 2:", empirical_distribution)
print("theoretical mu_0 P^2:             ", theoretical_distribution)


## Recap

1. Derive the quantile function for the cosine target and implement a second
   sampler by inversion. Compare it with the accept--reject sample.
2. In accept--reject sampling, explain why using an $M$ smaller than
   $\pi/2$ is invalid for the uniform proposal.
3. Create a three-state row-stochastic matrix, specify an initial
   distribution, and compare simulated time-$t$ frequencies with
   $\mu_0P^t$. State why a fresh input is required at each step.
